# دانلود ویدیوهای دوره سواد مالی

این نوت‌بوک پوشه `data/raw_videos/` را می‌سازد و ویدیوها را با کوکی‌های Chrome دانلود می‌کند. این پوشه در `.gitignore` است و وارد Git نمی‌شود.

> فقط ویدیوهایی را دانلود کنید که مجاز به دسترسی و نگهداری آن‌ها هستید. اگر لینک منقضی شد، لینک تازه `master.m3u8` را از Network جایگزین کنید.

In [ ]:
from pathlib import Path
import shutil
import subprocess

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
DOWNLOAD_DIR = PROJECT_ROOT / 'data' / 'raw_videos'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Download folder:', DOWNLOAD_DIR)


## لینک جلسات

برای هر جلسه، نام فایل و لینک تازه `master.m3u8` را در فهرست زیر قرار دهید.

In [ ]:
VIDEOS = [
    {
        'name': 'session-01',
        'url': 'https://lms.fintelligence.ir/video/play/farayad/62818/a6d37572c8d82cc7b4fca3914d9e39eb85295c96/master.m3u8',
    },
    {
        'name': 'session-02',
        'url': 'https://lms.fintelligence.ir/video/play/farayad/62813/0c667219800b39356d8f38863e7ea2b674608a03/master.m3u8',
    },
    # {'name': 'session-03', 'url': 'PASTE_FRESH_MASTER_M3U8_URL_HERE'},
]

len(VIDEOS)

In [ ]:
yt_dlp = shutil.which('yt-dlp')
if yt_dlp is None:
    local_binary = Path.home() / 'bin' / 'yt-dlp'
    if local_binary.exists():
        yt_dlp = str(local_binary)
    else:
        raise FileNotFoundError('yt-dlp پیدا نشد. فایل ~/bin/yt-dlp را نصب کنید.')

for video in VIDEOS:
    completed_file = DOWNLOAD_DIR / f"{video['name']}.mp4"
    if completed_file.is_file() and completed_file.stat().st_size > 0:
        print(f"Skipping {video['name']}: already downloaded ({completed_file.name})")
        continue

    output_template = str(DOWNLOAD_DIR / f"{video['name']}.%(ext)s")
    command = [
        yt_dlp,
        '--cookies-from-browser', 'chrome',
        '--referer', 'https://lms.fintelligence.ir/',
        '--merge-output-format', 'mp4',
        '--continue',
        '--no-overwrites',
        '-o', output_template,
        video['url'],
    ]
    print(f"\nDownloading {video['name']} ...")
    subprocess.run(command, check=True)

print('\nAll downloads finished.')


پس از پایان دانلود، فایل‌ها داخل `data/raw_videos/` هستند. برای بررسی:

```python
list(DOWNLOAD_DIR.glob('*'))
```

In [ ]:
# Cell 6 - Verify downloaded videos

downloaded_files = sorted(DOWNLOAD_DIR.glob('*'))

print('Downloaded files:', len(downloaded_files))
for file_path in downloaded_files:
    print(file_path.name)

In [ ]:
# Cell 7 - Inspect downloaded videos

import json

ffprobe = shutil.which('ffprobe')
if ffprobe is None:
    print('ffprobe نصب نیست؛ بررسی فنی ویدیوها رد شد. برای نصب در macOS: brew install ffmpeg')
else:
    for file_path in sorted(DOWNLOAD_DIR.glob('*.mp4')):
        command = [
            ffprobe,
            '-v', 'quiet',
            '-print_format', 'json',
            '-show_streams',
            '-show_format',
            str(file_path),
        ]
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode != 0:
            print(f'{file_path.name}: نامعتبر یا ناقص')
            continue
        metadata = json.loads(result.stdout)
        video_stream = next((s for s in metadata.get('streams', []) if s.get('codec_type') == 'video'), None)
        if video_stream is None:
            print(f'{file_path.name}: جریان ویدیو پیدا نشد')
            continue
        size_mb = int(metadata['format']['size']) / 1024 / 1024
        print(file_path.name)
        print('Resolution:', f"{video_stream['width']}x{video_stream['height']}")
        print('Size:', f'{size_mb:.2f} MB')
        print('Duration:', f"{float(metadata['format']['duration']) / 60:.1f} minutes")
        print()


In [ ]:
# Cell 8 - Extract course activity links

from bs4 import BeautifulSoup
from http.cookiejar import MozillaCookieJar
from pathlib import Path
import requests
import subprocess
import tempfile

COURSE_URL = 'https://lms.fintelligence.ir/course/view/12518'
cookie_file = Path(tempfile.gettempdir()) / 'fintelligence_cookies.txt'

export_command = [
    yt_dlp,
    '--cookies-from-browser', 'chrome',
    '--cookies', str(cookie_file),
    '--skip-download',
    COURSE_URL,
]

subprocess.run(export_command, check=False)

if not cookie_file.exists():
    raise FileNotFoundError('Cookie file was not created.')

cookie_jar = MozillaCookieJar(str(cookie_file))
cookie_jar.load(ignore_discard=True, ignore_expires=True)

session = requests.Session()
session.cookies = cookie_jar
session.headers.update({
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/140.0.0.0 Safari/537.36'
    )
})

response = session.get(
    COURSE_URL,
    timeout=30,
    allow_redirects=True,
)

print('Status:', response.status_code)
print('Final URL:', response.url)

if '/user/session' in response.url or '/login' in response.url:
    raise RuntimeError(
        'The website redirected to the login page. '
        'Open the course in Chrome, confirm that you are logged in, '
        'and run this cell again.'
    )

response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

activities = []
seen_urls = set()

selectors = [
    'a[href*="/activity/"]',
    'a[href*="/mod/"]',
    'a[href*="/video/"]',
]

for selector in selectors:
    for link in soup.select(selector):
        title = ' '.join(link.get_text(' ', strip=True).split())
        url = link.get('href')

        if not title or not url or url in seen_urls:
            continue

        seen_urls.add(url)
        activities.append({
            'title': title,
            'url': url,
        })

print('Activity links found:', len(activities))

for index, activity in enumerate(activities, start=1):
    print(f"\n{index:03d} | {activity['title']}")
    print(activity['url'])

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = 'https://lms.fintelligence.ir'
html_candidates = [CWD / 'course_page.html', PROJECT_ROOT / 'course_page.html']
HTML_FILE = next((path for path in html_candidates if path.is_file()), None)

if HTML_FILE is None:
    checked = '\n'.join(f'- {path}' for path in html_candidates)
    raise FileNotFoundError(f'course_page.html پیدا نشد. مسیرهای بررسی‌شده:\n{checked}')

with HTML_FILE.open('r', encoding='utf-8', errors='ignore') as file:
    soup = BeautifulSoup(file, 'html.parser')

activities = []
seen_urls = set()
for link in soup.select('a.activity-ajax'):
    href = link.get('href')
    if not href or href == '#':
        continue
    url = urljoin(BASE_URL, href)
    if url in seen_urls:
        continue
    seen_urls.add(url)
    activities.append({
        'id': link.get('data-id'),
        'title': link.get_text(' ', strip=True),
        'url': url,
    })

print('فایل:', HTML_FILE)
print('تعداد فعالیت‌ها:', len(activities))
for index, activity in enumerate(activities, start=1):
    print(f"{index}. {activity['title']}")
    print(activity['url'])
